# PatchTST + Latent Channel Mixing on Electricity (deterministic interpolation)

This notebook tests whether PatchTST's strict channel independence can be relaxed by adding **one lightweight latent cross-channel attention layer** between the per-channel encoder and the forecasting head.

**What we are NOT doing:**
- We are not running paper's original "channel-mixing" baseline (which mixes channels at the *raw timestep* token level).
- We are not claiming our results are directly comparable to paper Table 3 absolute MSE.

**What we ARE doing — CI-first latent mixing extension:**
1. PatchTST first learns per-channel temporal patch representations (channel-independent encoder, paper unchanged).
2. **After** the encoder, we optionally apply a single cross-channel attention layer over the M channels at each patch position.
3. The strength of this latent mixing is controlled by `mix_alpha ∈ [0, 1]`:
   - `mix_alpha = 0.0` → pure PatchTST CI baseline (mixer never used).
   - `mix_alpha = 1.0` → encoder output entirely replaced by mixed representation.
   - `mix_alpha ∈ (0, 1)` → linear interpolation `z + alpha * (z_mix - z)`.

**Design choice (important):** the mixer is **deterministically** computed on every forward pass whenever `mix_alpha > 0`, then linearly interpolated. This avoids the confound where stochastic gating (only sometimes calling the mixer) would also limit how much training signal the mixer receives.

**Main question:**

> Does latent cross-channel attention added on top of CI improve forecasting on Electricity, and how does the gain scale with mixing strength?


## 0. Install dependencies

In [ ]:
# (Skip if Colab already has these.)


## 1. Imports, seed, and device

In [ ]:
import os, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader


def set_seed(seed=2021):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(2021)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Configuration — Electricity dataset

Set `DATA_CSV_PATH` to your Electricity CSV (e.g., the standard PatchTST benchmark Electricity dataset). The loader keeps numeric columns only, so the timestamp column is automatically dropped.


In [ ]:
# === CHANGE 1: dataset is Electricity, path renamed accordingly ===
from google.colab import drive
drive.mount('/content/drive')

DATA_CSV_PATH = "/content/drive/MyDrive/p/electricity.csv"
print("DATA_CSV_PATH:", DATA_CSV_PATH)
print("Exists:", os.path.exists(DATA_CSV_PATH))


In [ ]:
# === CHANGE 1: OUTPUT_DIR renamed weather → electricity ===
CFG = {
    'seq_len': 96,
    'forecast_len': 96,
    'patch_size': 16,
    'stride': 8,
    'd_model': 64,
    'nhead': 4,
    'num_layers': 3,
    'dim_feedforward': 128,
    'dropout': 0.1,

    'batch_size': 8,         # important: lower for 321-channel Electricity
    'epochs': 5,
    'patience': 2,
    'learning_rate': 1e-3,

    'train_ratio': 0.7,
    'valid_ratio': 0.1,

    'step': 4,                # important: fewer sliding windows, much faster
    'seed': 2021,
}

# === CHANGE: argument naming clarification ===
# We keep the variable name MIX_PROBS for backward compatibility with the rest of
# the notebook, but conceptually these values are now interpreted as MIXING ALPHAS
# (deterministic interpolation strengths), not stochastic activation probabilities.
MIX_PROBS = [0.0, 0.3, 0.5, 1.0]

OUTPUT_DIR = Path('./scm_electricity_outputs')   # ← renamed
OUTPUT_DIR.mkdir(exist_ok=True)

CFG


## 3. Data loading and sliding windows

Numeric columns only (timestamp dropped). Train-set mean/std applied to all splits (no leakage).

**Caveat (acknowledge in writeup):** we apply train z-score normalization at the dataloader level AND RevIN inside the model (per-instance). This is a conservative double-normalization. It is matched across all `mix_alpha` values, so relative comparisons are fair, but absolute MSE is not directly comparable to paper.


In [ ]:
# === CHANGE 2: function renamed load_weather_csv → load_timeseries_csv ===
# === CHANGE 3: DATA_FRACTION = 1 → 1.0 (style) ===
def load_timeseries_csv(csv_path, max_channels=None):
    if csv_path is None or not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"Could not find dataset CSV at {csv_path}. "
            "Set DATA_CSV_PATH to the correct file (e.g., electricity.csv). "
            "For Colab, upload it or mount Google Drive."
        )
    df = pd.read_csv(csv_path)
    print('Raw shape:', df.shape)
    print('Columns:', list(df.columns)[:10], '...')

    numeric_df = df.select_dtypes(include=[np.number]).copy()
    if numeric_df.shape[1] == 0:
        raise ValueError('No numeric columns found. Check the CSV format.')

    if max_channels is not None:
        numeric_df = numeric_df.iloc[:, :max_channels]

    numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan).ffill().bfill()
    print('Numeric feature shape:', numeric_df.shape)
    return numeric_df.astype(np.float32)


def create_dataloaders_from_array(data, seq_len, forecast_len, batch_size=32,
                                  train_ratio=0.7, valid_ratio=0.1, step=1):
    T_total, M = data.shape
    train_end = int(train_ratio * T_total)
    valid_end = int((train_ratio + valid_ratio) * T_total)

    train_raw = data[:train_end]
    mean = train_raw.mean(axis=0, keepdims=True)
    std = train_raw.std(axis=0, keepdims=True) + 1e-8

    train_data = (data[:train_end] - mean) / std
    valid_data = (data[train_end:valid_end] - mean) / std
    test_data  = (data[valid_end:] - mean) / std

    def sliding_window(d):
        X, y = [], []
        total = seq_len + forecast_len
        for i in range(0, len(d) - total + 1, step):
            X.append(d[i:i+seq_len])
            y.append(d[i+seq_len:i+total])
        if len(X) == 0:
            raise ValueError('Not enough data for the chosen seq_len + forecast_len.')
        return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

    X_train, y_train = sliding_window(train_data)
    X_valid, y_valid = sliding_window(valid_data)
    X_test, y_test = sliding_window(test_data)

    print(f'\nData split:')
    print(f'  total timesteps = {T_total}, variables = {M}')
    print(f'  train/valid/test timesteps = {train_end}/{valid_end-train_end}/{T_total-valid_end}')
    print(f'\nWindow shapes:')
    print(f'  train X={tuple(X_train.shape)}, y={tuple(y_train.shape)}')
    print(f'  valid X={tuple(X_valid.shape)}, y={tuple(y_valid.shape)}')
    print(f'  test  X={tuple(X_test.shape)}, y={tuple(y_test.shape)}')

    return (
        DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True),
        DataLoader(TensorDataset(X_valid, y_valid), batch_size=batch_size, shuffle=False),
        DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False),
        {'mean': mean, 'std': std, 'num_variables': M},
    )

# === CHANGE 2: function call renamed ===
data_df = load_timeseries_csv(DATA_CSV_PATH)
data = data_df.to_numpy(dtype=np.float32)

# === CHANGE 3: explicit float ===
DATA_FRACTION = 1.0
T_use = int(len(data) * DATA_FRACTION)
data = data[:T_use]

print("Using timesteps:", data.shape[0])
print("Using channels:", data.shape[1])

train_loader, valid_loader, test_loader, data_info = create_dataloaders_from_array(
    data=data,
    seq_len=CFG['seq_len'],
    forecast_len=CFG['forecast_len'],
    batch_size=CFG['batch_size'],
    train_ratio=CFG['train_ratio'],
    valid_ratio=CFG['valid_ratio'],
    step=CFG['step'],
)
NUM_VARIABLES = data_info['num_variables']
print('NUM_VARIABLES =', NUM_VARIABLES)


## 4. PatchTST baseline with explicit `encode` and `decode`

Channel-independent PatchTST: `(B, L, M) → (B*M, L)`. Each channel passes through the encoder as a separate univariate series with shared weights. We expose `encode()` and `decode()` so the mixer can be inserted between them without modifying the backbone.


In [ ]:
class RevIN(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(1))
        self.affine_bias = nn.Parameter(torch.zeros(1))
        self.mean = None
        self.std = None

    def normalize(self, x):
        # x: (B*M, L)
        self.mean = x.mean(dim=-1, keepdim=True)
        self.std = x.std(dim=-1, keepdim=True) + self.eps
        x = (x - self.mean) / self.std
        return x * self.affine_weight + self.affine_bias

    def denormalize(self, x):
        # x: (B*M, T)
        return ((x - self.affine_bias) / (self.affine_weight + self.eps)) * self.std + self.mean


class PatchEmbedding(nn.Module):
    def __init__(self, patch_size, d_model, num_patches):
        super().__init__()
        self.Wp = nn.Linear(patch_size, d_model, bias=False)
        self.Wpos = nn.Parameter(torch.randn(1, num_patches, d_model) * 0.02)

    def forward(self, x_p):
        return self.Wp(x_p) + self.Wpos


class BatchNormTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ff1 = nn.Linear(d_model, dim_feedforward)
        self.ff2 = nn.Linear(dim_feedforward, d_model)
        self.dropout = nn.Dropout(dropout)
        self.bn1 = nn.BatchNorm1d(d_model)
        self.bn2 = nn.BatchNorm1d(d_model)

    def forward(self, x):
        attn_out, _ = self.self_attn(x, x, x, need_weights=False)
        x = x + self.dropout(attn_out)
        x = self.bn1(x.transpose(1, 2)).transpose(1, 2)
        ff_out = self.ff2(self.dropout(F.gelu(self.ff1(x))))
        x = x + self.dropout(ff_out)
        x = self.bn2(x.transpose(1, 2)).transpose(1, 2)
        return x


class PatchTST_CI(nn.Module):
    def __init__(self, seq_len, forecast_len, patch_size, stride,
                 d_model, nhead=4, num_layers=3, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.forecast_len = forecast_len
        self.patch_size = patch_size
        self.stride = stride
        self.d_model = d_model
        self.num_patches = math.floor((seq_len - patch_size) / stride) + 2

        self.revin = RevIN()
        self.patch_embed = PatchEmbedding(patch_size, d_model, self.num_patches)
        self.encoder_layers = nn.ModuleList([
            BatchNormTransformerLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.flatten = nn.Flatten(start_dim=1)
        self.linear_head = nn.Linear(self.num_patches * d_model, forecast_len)

    def _make_patches(self, x):
        x = F.pad(x, (0, self.stride), mode='replicate')
        return x.unfold(dimension=-1, size=self.patch_size, step=self.stride)

    def encode(self, x):
        # x: (B, L, M)
        B, L, M = x.shape
        x_uni = x.permute(0, 2, 1).reshape(B * M, L)
        x_uni = self.revin.normalize(x_uni)
        z = self._make_patches(x_uni)
        z = self.patch_embed(z)
        for layer in self.encoder_layers:
            z = layer(z)
        return z, B, M

    def decode(self, z, B, M):
        out = self.linear_head(self.flatten(z))
        out = self.revin.denormalize(out)
        return out.reshape(B, M, self.forecast_len)

    def forward(self, x):
        z, B, M = self.encode(x)
        return self.decode(z, B, M)


## 5. Latent Cross-Channel Mixer + DIM model

**Cross-channel attention** is applied across the M channels at each patch position. Reshape `(B*M, P, D) → (B*P, M, D)`, attend across M, reshape back.

### Why deterministic interpolation, not stochastic gating?

The earlier draft used a stochastic version: with probability `mix_prob`, replace `z` with `mixer(z)`. This conflated two distinct effects:

1. **Mixing strength**: how much cross-channel info is injected.
2. **Mixer training frequency**: how many batches actually backprop through the mixer.

Low `mix_prob` (e.g., 0.1) starves the mixer of training signal — so the comparison `mix_prob=0.1` vs `mix_prob=0.9` is partly "barely-trained mixer" vs "well-trained mixer", not just "weak mixing" vs "strong mixing".

**Deterministic Interpolation Mixing (DIM)** fixes this: when `mix_alpha > 0`, the mixer is **always** computed (so it always gets gradient), and the output is interpolated:

```
z_out = z + mix_alpha * (z_mix - z) = (1 - mix_alpha) * z + mix_alpha * z_mix
```

Now `mix_alpha` cleanly controls mixing strength only. Mixer training is independent of `mix_alpha` (always trained when `> 0`).


In [ ]:
class CrossChannelMixingLayer(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z, B, M):
        # z: (B*M, P, D)
        BM, P, D = z.shape
        assert BM == B * M, f'Expected BM={B*M}, got {BM}'

        # (B*M, P, D) -> (B, M, P, D) -> (B*P, M, D)
        z_chan = z.reshape(B, M, P, D).permute(0, 2, 1, 3).reshape(B * P, M, D)
        attn_out, _ = self.attn(z_chan, z_chan, z_chan, need_weights=False)
        z_chan = self.norm(z_chan + self.dropout(attn_out))

        # (B*P, M, D) -> (B*M, P, D)
        z_out = z_chan.reshape(B, P, M, D).permute(0, 2, 1, 3).reshape(B * M, P, D)
        return z_out


# === CHANGE 4: PatchTST_SCM rewritten as Deterministic Interpolation Mixing ===
class PatchTST_SCM(nn.Module):
    """PatchTST + DETERMINISTIC LATENT CHANNEL MIXING (DIM).

    mix_alpha (named mix_prob for back-compat with sweep cells):
        0.0      → pure PatchTST CI baseline (mixer is never used; gradient never flows)
        1.0      → encoder output entirely replaced by mixed representation
        in (0,1) → linear interpolation z + alpha*(z_mix - z)

    The mixer is COMPUTED ON EVERY FORWARD PASS whenever mix_alpha > 0.
    Therefore alpha cleanly controls mixing STRENGTH and is decoupled from
    how often the mixer is trained.
    """
    def __init__(self, seq_len, forecast_len, patch_size, stride,
                 d_model, nhead=4, num_layers=3, dim_feedforward=128, dropout=0.1,
                 mix_prob=0.0, mix_heads=4):
        super().__init__()
        # name kept as mix_prob for sweep-cell back-compat; semantically it's alpha now
        self.mix_alpha = float(mix_prob)
        self.backbone = PatchTST_CI(
            seq_len=seq_len, forecast_len=forecast_len,
            patch_size=patch_size, stride=stride,
            d_model=d_model, nhead=nhead, num_layers=num_layers,
            dim_feedforward=dim_feedforward, dropout=dropout,
        )
        self.cross_mixer = CrossChannelMixingLayer(
            d_model=d_model, nhead=mix_heads, dropout=dropout,
        )

    def forward(self, x):
        z, B, M = self.backbone.encode(x)
        if self.mix_alpha <= 0:
            z_out = z
        else:
            # Always compute mixer → always gets gradient; alpha is pure strength.
            z_mix = self.cross_mixer(z, B, M)
            z_out = z + self.mix_alpha * (z_mix - z)
        return self.backbone.decode(z_out, B, M)


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# Shape sanity check
x0, y0 = next(iter(train_loader))
mini_model = PatchTST_SCM(
    **{k: CFG[k] for k in ['seq_len','forecast_len','patch_size','stride',
                           'd_model','nhead','num_layers','dim_feedforward','dropout']},
    mix_prob=0.5,
).to(device)
with torch.no_grad():
    out0 = mini_model(x0[:2].to(device))
print('Input shape:', tuple(x0[:2].shape))
print('Output shape:', tuple(out0.shape), 'Expected:', (2, NUM_VARIABLES, CFG['forecast_len']))
print('Params:', count_parameters(mini_model))
del mini_model
if device == 'cuda':
    torch.cuda.empty_cache()


## 6. Training and evaluation utilities


In [ ]:
def evaluate(model, loader, device, return_predictions=False):
    model.eval()
    criterion = nn.MSELoss()
    total_loss = 0.0
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.permute(0, 2, 1).to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            total_loss += loss.item() * xb.size(0)
            if return_predictions:
                preds.append(pred.detach().cpu())
                trues.append(yb.detach().cpu())
    avg_loss = total_loss / len(loader.dataset)
    if return_predictions:
        return avg_loss, torch.cat(preds, dim=0).numpy(), torch.cat(trues, dim=0).numpy()
    return avg_loss


def train_one_model(model, train_loader, valid_loader, device,
                    epochs=8, lr=1e-3, patience=3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = {'train_loss': [], 'valid_loss': []}
    best_valid = float('inf')
    best_state = None
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False)
        for xb, yb in pbar:
            xb = xb.to(device)
            yb = yb.permute(0, 2, 1).to(device)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * xb.size(0)
            pbar.set_postfix({'train_mse': f'{loss.item():.4f}'})

        train_loss = running / len(train_loader.dataset)
        valid_loss = evaluate(model, valid_loader, device)
        history['train_loss'].append(train_loss)
        history['valid_loss'].append(valid_loss)
        print(f'Epoch {epoch:02d}: train={train_loss:.6f}, valid={valid_loss:.6f}')

        if valid_loss < best_valid - 1e-7:
            best_valid = valid_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f'Early stopping at epoch {epoch}. Best valid={best_valid:.6f}')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_valid


## 7. Run the alpha sweep on Electricity

**IMPORTANT:** if you change `step`, `batch_size`, `train_ratio`, `valid_ratio`, or `DATA_FRACTION`, you must **re-run the data-loader cell** before this sweep. Sliding windows are baked into `train_loader` at creation time.


In [ ]:
# Sweep config — finer resolution for the headline figure
CFG['epochs'] = 20
CFG['patience'] = 3
CFG['step'] = 4   # consistent with cell that built the dataloaders above

# === CHANGE: variable kept as MIX_PROBS for back-compat,
# but interpreted as MIX_ALPHAS (deterministic interpolation strengths) ===
MIX_PROBS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


In [ ]:
results = []
histories = {}
models = {}

base_kwargs = {k: CFG[k] for k in ['seq_len','forecast_len','patch_size','stride',
                                    'd_model','nhead','num_layers','dim_feedforward','dropout']}

for p in MIX_PROBS:
    print('\n' + '='*80)
    print(f'Training PatchTST_SCM (DIM) with mix_alpha = {p}')
    print('='*80)
    set_seed(CFG['seed'])

    model = PatchTST_SCM(**base_kwargs, mix_prob=p, mix_heads=CFG['nhead'])
    total_params, trainable_params = count_parameters(model)
    print(f'Parameters: total={total_params:,}, trainable={trainable_params:,}')

    start = time.time()
    model, history, best_valid = train_one_model(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        device=device,
        epochs=CFG['epochs'],
        lr=CFG['learning_rate'],
        patience=CFG['patience'],
    )
    elapsed_min = (time.time() - start) / 60
    test_mse, preds, targets = evaluate(model, test_loader, device, return_predictions=True)

    row = {
        'mix_alpha': p,                  # === CHANGE: column renamed to mix_alpha
        'best_valid_mse': best_valid,
        'test_mse': test_mse,
        'test_rmse': float(np.sqrt(test_mse)),
        'params': total_params,
        'elapsed_min': elapsed_min,
    }
    print('RESULT:', row)
    results.append(row)
    histories[p] = history
    models[p] = model.cpu()

    np.savez_compressed(
        OUTPUT_DIR / f'preds_alpha_{str(p).replace(".", "p")}.npz',
        preds=preds, targets=targets,
    )

    if device == 'cuda':
        torch.cuda.empty_cache()

# === CHANGE 1: filename renamed weather → electricity ===
results_df = pd.DataFrame(results).sort_values('mix_alpha')
results_df.to_csv(OUTPUT_DIR / 'results_scm_electricity.csv', index=False)
results_df


## 8. Plot results

Headline figure: `mix_alpha` vs test MSE. Validation curves overlay confirms whether high-alpha variants overfit or not.


In [ ]:
# === CHANGE 1: title renamed Weather → Electricity ===
plt.figure(figsize=(7, 4))
plt.plot(results_df['mix_alpha'], results_df['test_mse'], marker='o')
plt.xlabel('mix_alpha (deterministic interpolation strength)')
plt.ylabel('Test MSE')
plt.title('Electricity: PatchTST + Deterministic Interpolation Mixing (DIM)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mix_alpha_vs_test_mse.png', dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
for p, hist in histories.items():
    plt.plot(hist['valid_loss'], marker='o', label=f'α={p}')
plt.xlabel('Epoch')
plt.ylabel('Validation MSE')
plt.title('Electricity: validation curves by mix_alpha')
plt.legend(ncol=2, fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'validation_curves.png', dpi=200)
plt.show()

print('Saved outputs to:', OUTPUT_DIR.resolve())
print(results_df)


## 9. Forecast visualization for the best model


In [ ]:
best_row = results_df.loc[results_df['test_mse'].idxmin()]
best_alpha = float(best_row['mix_alpha'])
print('Best mix_alpha by test MSE:', best_alpha)

pred_file = OUTPUT_DIR / f'preds_alpha_{str(best_alpha).replace(".", "p")}.npz'
arr = np.load(pred_file)
preds = arr['preds']
targets = arr['targets']

sample_idx = 0
num_vars_to_plot = min(4, preds.shape[1])
for var_idx in range(num_vars_to_plot):
    plt.figure(figsize=(7, 3))
    plt.plot(targets[sample_idx, var_idx], label='Target')
    plt.plot(preds[sample_idx, var_idx], label='Prediction')
    plt.xlabel('Forecast step')
    plt.ylabel('Standardized value')
    plt.title(f'Best DIM model α={best_alpha}: sample {sample_idx}, variable {var_idx}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'forecast_best_alpha_{best_alpha}_var_{var_idx}.png', dpi=200)
    plt.show()


## 10. Presentation-ready interpretation template (revised — bulletproof framing)

### What we measured

We swept `mix_alpha ∈ {0.0, 0.1, ..., 1.0}` controlling the interpolation strength of a single latent cross-channel attention layer added on top of PatchTST_CI's encoder. The mixer is computed deterministically every forward pass (so its training signal is independent of α).

### Choose the branch that matches your actual result

**Case A — middle `mix_alpha` wins**

> The optimum occurs at an intermediate alpha, supporting a regularization-vs-expressiveness tradeoff: pure CI (α=0) under-uses cross-channel information, while full mix replacement (α=1) loses some of CI's regularization. Best practice for Electricity-like multivariate forecasting is therefore not to choose between CI and CM but to interpolate.

**Case B — `mix_alpha = 0.0` wins**

> Pure PatchTST CI remains the strongest configuration on Electricity in our setup. This is consistent with the paper's argument (Appendix A.7) that channel-independence acts as a useful architectural regularizer on multivariate panels with limited cross-channel predictive structure beyond what shared encoder weights already capture.

**Case C — `mix_alpha = 1.0` (or high) wins**

> Full deterministic mixing yields the lowest test MSE. This contradicts the paper's claim (Appendix A.7) that channel-mixing causes overfitting — our validation curves show the high-α variant continues to improve through training. We attribute this to the design being a CI-FIRST hybrid: the per-channel encoder still learns temporal patterns; mixing operates only on latent representations, not on raw inputs. Channel independence should therefore be reframed not as a fixed truth but as one extreme of a continuous mixing spectrum.

### Mandatory caveats to include in the writeup

1. **Not paper's CM baseline.** Our SCM/DIM is a *CI-first latent mixing extension*, not the paper's "channel-mixing" baseline (which mixes at raw timestep token level). We are **not** claiming to reproduce or contradict paper's Table 7 CM number directly.

2. **mix_alpha is interpolation strength, NOT mixing probability.** In an earlier draft we used stochastic gating, which conflated mixing strength with mixer training frequency. The current implementation uses deterministic interpolation, so α cleanly controls strength only.

3. **Double normalization.** All variants use both train z-score normalization (data-loader level) and RevIN (model level). The comparison across α is therefore matched, but our absolute MSE is not directly comparable to paper's Table 3.

4. **Reduced training budget.** We use a smaller model (d_model=64 vs paper's 128) and fewer epochs (20 vs 100) to keep the 11-point sweep tractable. Relative ranking is the finding, not absolute MSE.

### Safe headline sentence

> Treating channel-independence as a continuous interpolation parameter rather than a binary choice, our deterministic interpolation mixing experiment on Electricity finds optimal mix_alpha = X (test MSE Y), [reframe / confirm / contradict paper's CI advocacy] under matched preprocessing and training budget.


## 11. Save experiment metadata


In [ ]:
# === CHANGE 8: metadata key renamed; explicit dataset field added ===
metadata = {
    'dataset': 'Electricity',         # NEW
    'data_csv_path': DATA_CSV_PATH,    # was 'weather_csv_path'
    'cfg': CFG,
    'mix_alphas': MIX_PROBS,           # renamed key (semantically alpha)
    'num_variables': int(NUM_VARIABLES),
    'mixing_design': 'deterministic_interpolation',
    'mixer_position': 'post_encoder_pre_head',
    'normalization': 'train_zscore + RevIN (matched across all alpha)',
    'results': results_df.to_dict(orient='records'),
}
with open(OUTPUT_DIR / 'experiment_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print('Metadata saved to:', OUTPUT_DIR / 'experiment_metadata.json')
print('Results CSV:', OUTPUT_DIR / 'results_scm_electricity.csv')
print('Main plot:', OUTPUT_DIR / 'mix_alpha_vs_test_mse.png')
